# Task 3 overnight sweep (~4 hours)

"Run All" once. The sweeps run one after another in order of importance; **no new run starts after the deadline**
(4 hours after the setup cell). A failing sweep is reported and skipped, the others continue.
Every sweep writes its own file to `output/task3/`; the last cell prints the overview.

Order (estimated run times on the RTX 5080 from the Task 3 training on 7 Nov, without the refit):
1. shared baseline = bias model (closed form, < 1 s per run)
2. time-aware bias model: temporal bias terms (closed form, seconds)
3. item kNN and time-decayed kNN (~10 - 20 s per run)
4. zero model, then the time zero model with the zero model's best priors (they enter the blend as a pair)
5. SVD++ and timeSVD++ (bias / factor regularization, learning rate; ~20 - 60 s per run)
6. SVD (k = 500, small batches: ~2 min per run)
7. refinements: SVD++ / SVD factor counts around the winners of step 5 / 6
8. until the deadline: random search for timeSVD++ (the strongest factor model in the Task 3 blend)

Closed-form and kNN sweeps re-check their 3 best combinations on split seeds 1 and 2 (cheap); the factor models do it
for their 2 best. Nothing is written to `lib/config.py` or `trained_models/` -> pick the winners afterwards.

In [1]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta

import torch

import grid_search
from grid_search import print_sweep_overview, run_grid_search

DEADLINE = datetime.now() + timedelta(hours=4)
print("deadline", f"{DEADLINE:%Y-%m-%d %H:%M}", "|", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")


# one sweep of the night: skipped after the deadline, an error is printed instead of stopping "Run All"
def overnight_sweep(model_name: str, parameter_grid: dict, **options):
    if datetime.now() >= DEADLINE:
        print(f"{model_name}: skipped (deadline reached)")
        return None
    try:
        return run_grid_search("task3", model_name, parameter_grid, deadline=DEADLINE, **options)
    except Exception as error:
        print(f"{model_name}: sweep failed -> skipped ({type(error).__name__}: {error})")
        grid_search.free_memory()
        return None
    finally:
        grid_search.keep_computer_awake(True)   # stay awake between the sweeps too


# swept values of a sweep's winner (empty if the sweep was skipped or failed) -> constants of a follow-up sweep
def best_values(report) -> dict:
    return dict(report["best"]["swept_values"]) if report and report.get("best") else {}


grid_search.keep_computer_awake(True)

deadline 2026-11-08 13:58 | 2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


## 1) Shared baseline (bias model, item kNN, time kNN)

In [2]:
bias_model_sweep = overnight_sweep("bias_model", {
    "user_bias_regularization": [0.5, 1.0, 2.0, 3.0, 5.0],
    "item_bias_regularization": [2.0, 3.0, 5.0, 8.0, 12.0],
    "n_iterations": 50,
}, rerun_finalists_on_other_splits=True)

task3 / bias_model: 25 combinations + 6 finalist runs on split seeds [1, 2] | swept: {'user_bias_regularization': [0.5, 1.0, 2.0, 3.0, 5.0], 'item_bias_regularization': [2.0, 3.0, 5.0, 8.0, 12.0]} | constant: {'n_iterations': 50}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\bias_model_2026-11-08_09-58-29.json
[1/25] user_bias_regularization=0.5 item_bias_regularization=2 ...
loading task3 with validation split seed 0 ...
    validation MSE 0.91600 | 1 s | best so far 0.91600
[2/25] user_bias_regularization=0.5 item_bias_regularization=3 ...
    validation MSE 0.91557 | 1 s | best so far 0.91557
[3/25] user_bias_regularization=0.5 item_bias_regularization=5 ...
    validation MSE 0.91525 | 1 s | best so far 0.91525
[4/25] user_bias_regularization=0.5 item_bias_regularization=8 ...
    validation MSE 0.91550 | 1 s | best so far 0.91525
[5/25] user_bias_regularization=0.5 item_bias_regularization=12 ...
    validation MSE 0.91646

## 2) Time-aware bias model (temporal bias terms, shared with timeSVD++)

In [3]:
time_bias_model_sweep = overnight_sweep("time_bias_model", {
    "temporal_biases.time_bias_regularization": [25.0, 50.0, 100.0],
    "temporal_biases.drift_regularization": [1000.0, 3000.0, 10000.0],
    "bias_regularization": [1.0, 3.0, 5.0],
}, rerun_finalists_on_other_splits=True)

task3 / time_bias_model: 27 combinations + 6 finalist runs on split seeds [1, 2] | swept: {'temporal_biases.time_bias_regularization': [25.0, 50.0, 100.0], 'temporal_biases.drift_regularization': [1000.0, 3000.0, 10000.0], 'bias_regularization': [1.0, 3.0, 5.0]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\time_bias_model_2026-11-08_09-58-51.json
[1/27] temporal_biases.time_bias_regularization=25 temporal_biases.drift_regularization=1000 bias_regularization=1 ...
    validation MSE 0.88803 | 3 s | best so far 0.88803
[2/27] temporal_biases.time_bias_regularization=25 temporal_biases.drift_regularization=1000 bias_regularization=3 ...
    validation MSE 0.88686 | 3 s | best so far 0.88686
[3/27] temporal_biases.time_bias_regularization=25 temporal_biases.drift_regularization=1000 bias_regularization=5 ...
    validation MSE 0.88871 | 3 s | best so far 0.88686
[4/27] temporal_biases.time_bias_regularization=25 temp

## 3) Item kNN and time-decayed item kNN

In [4]:
item_knn_sweep = overnight_sweep("item_knn", {
    "n_neighbors": [50, 100, 200],
    "prediction_shrinkage": [0.3, 1.0, 3.0],
    "similarity_shrinkage": [50.0, 100.0, 200.0],
}, rerun_finalists_on_other_splits=True)

task3 / item_knn: 27 combinations + 6 finalist runs on split seeds [1, 2] | swept: {'n_neighbors': [50, 100, 200], 'prediction_shrinkage': [0.3, 1.0, 3.0], 'similarity_shrinkage': [50.0, 100.0, 200.0]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\item_knn_2026-11-08_10-00-13.json
[1/27] n_neighbors=50 prediction_shrinkage=0.3 similarity_shrinkage=50 ...
    validation MSE 0.88436 | 6 s | best so far 0.88436
[2/27] n_neighbors=50 prediction_shrinkage=0.3 similarity_shrinkage=100 ...
    validation MSE 0.88667 | 5 s | best so far 0.88436
[3/27] n_neighbors=50 prediction_shrinkage=0.3 similarity_shrinkage=200 ...
    validation MSE 0.88979 | 6 s | best so far 0.88436
[4/27] n_neighbors=50 prediction_shrinkage=1 similarity_shrinkage=50 ...
    validation MSE 0.89753 | 5 s | best so far 0.88436
[5/27] n_neighbors=50 prediction_shrinkage=1 similarity_shrinkage=100 ...
    validation MSE 0.89941 | 5 s | best so far 0.88

In [5]:
time_item_knn_sweep = overnight_sweep("time_item_knn", {
    "time_decay_days": [90.0, 365.0, 1000.0],
    "time_decay_floor": [0.3, 0.5, 0.7],
    "n_neighbors": [50, 100],
}, rerun_finalists_on_other_splits=True)

task3 / time_item_knn: 18 combinations + 6 finalist runs on split seeds [1, 2] | swept: {'time_decay_days': [90.0, 365.0, 1000.0], 'time_decay_floor': [0.3, 0.5, 0.7], 'n_neighbors': [50, 100]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\time_item_knn_2026-11-08_10-03-19.json
[1/18] time_decay_days=90 time_decay_floor=0.3 n_neighbors=50 ...
    validation MSE 0.90215 | 6 s | best so far 0.90215
[2/18] time_decay_days=90 time_decay_floor=0.3 n_neighbors=100 ...
    validation MSE 0.90204 | 6 s | best so far 0.90204
[3/18] time_decay_days=90 time_decay_floor=0.5 n_neighbors=50 ...
    validation MSE 0.90125 | 6 s | best so far 0.90125
[4/18] time_decay_days=90 time_decay_floor=0.5 n_neighbors=100 ...
    validation MSE 0.90115 | 6 s | best so far 0.90115
[5/18] time_decay_days=90 time_decay_floor=0.7 n_neighbors=50 ...
    validation MSE 0.90046 | 6 s | best so far 0.90046
[6/18] time_decay_days=90 time_decay_floo

## 4) Zero model, then the time zero model with the same priors

In [6]:
zero_model_sweep = overnight_sweep("zero_model", {
    "user_prior_strength": [2.0, 5.0, 10.0],
    "item_prior_strength": [10.0, 20.0, 40.0],
    "inverse_regularization": [0.3, 1.0, 3.0],
})

task3 / zero_model: 27 combinations | swept: {'user_prior_strength': [2.0, 5.0, 10.0], 'item_prior_strength': [10.0, 20.0, 40.0], 'inverse_regularization': [0.3, 1.0, 3.0]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\zero_model_2026-11-08_10-05-36.json
[1/27] user_prior_strength=2 item_prior_strength=10 inverse_regularization=0.3 ...
    validation log loss of P(0) 0.05714 | 0 s | best so far 0.05714
[2/27] user_prior_strength=2 item_prior_strength=10 inverse_regularization=1 ...
    validation log loss of P(0) 0.05714 | 0 s | best so far 0.05714
[3/27] user_prior_strength=2 item_prior_strength=10 inverse_regularization=3 ...
    validation log loss of P(0) 0.05714 | 0 s | best so far 0.05714
[4/27] user_prior_strength=2 item_prior_strength=20 inverse_regularization=0.3 ...
    validation log loss of P(0) 0.05714 | 0 s | best so far 0.05714
[5/27] user_prior_strength=2 item_prior_strength=20 inverse_regularizati

In [7]:
time_zero_model_sweep = overnight_sweep("time_zero_model", {
    **best_values(zero_model_sweep),                # the pair must share its priors
    "window_prior_strength": [0.5, 1.0, 2.0, 5.0],
})

task3 / time_zero_model: 4 combinations | swept: {'window_prior_strength': [0.5, 1.0, 2.0, 5.0]} | constant: {'user_prior_strength': 10.0, 'item_prior_strength': 20.0, 'inverse_regularization': 3.0}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\time_zero_model_2026-11-08_10-05-48.json
[1/4] window_prior_strength=0.5 ...
    validation log loss of P(0) 0.05378 | 1 s | best so far 0.05378
[2/4] window_prior_strength=1 ...
    validation log loss of P(0) 0.05379 | 1 s | best so far 0.05378
[3/4] window_prior_strength=2 ...
    validation log loss of P(0) 0.05380 | 1 s | best so far 0.05378
[4/4] window_prior_strength=5 ...
    validation log loss of P(0) 0.05382 | 1 s | best so far 0.05378
best: window_prior_strength=0.5 -> validation log loss of P(0) 0.05378


## 5) SVD++ and timeSVD++

In [8]:
svdpp_sweep = overnight_sweep("svdpp", {
    "user_bias_regularization": [1.0, 3.0],
    "item_bias_regularization": [3.0, 5.0],
    "factor_regularization": [4.0, 8.0, 16.0],
    "learning_rate": [1e-3, 2e-3],
}, rerun_finalists_on_other_splits=True, n_finalists=2)

task3 / svdpp: 24 combinations + 4 finalist runs on split seeds [1, 2] | swept: {'user_bias_regularization': [1.0, 3.0], 'item_bias_regularization': [3.0, 5.0], 'factor_regularization': [4.0, 8.0, 16.0], 'learning_rate': [0.001, 0.002]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\svdpp_2026-11-08_10-05-52.json
[1/24] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=4 learning_rate=0.001 ...
    validation MSE 0.88634 (epoch 3) | 29 s | best so far 0.88634
[2/24] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=4 learning_rate=0.002 ...
    validation MSE 0.88801 (epoch 2) | 25 s | best so far 0.88634
[3/24] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=8 learning_rate=0.001 ...
    validation MSE 0.88671 (epoch 3) | 28 s | best so far 0.88634
[4/24] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=8 lea

In [9]:
time_svdpp_sweep = overnight_sweep("time_svdpp", {
    "factor_regularization": [4.0, 8.0, 16.0],
    "learning_rate": [1e-3, 2e-3],
    "bias_regularization": [1.0, 3.0],
}, rerun_finalists_on_other_splits=True, n_finalists=2)

task3 / time_svdpp: 12 combinations + 4 finalist runs on split seeds [1, 2] | swept: {'factor_regularization': [4.0, 8.0, 16.0], 'learning_rate': [0.001, 0.002], 'bias_regularization': [1.0, 3.0]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\time_svdpp_2026-11-08_10-18-43.json
[1/12] factor_regularization=4 learning_rate=0.001 bias_regularization=1 ...
    validation MSE 0.85855 (epoch 3) | 30 s | best so far 0.85855
[2/12] factor_regularization=4 learning_rate=0.001 bias_regularization=3 ...
    validation MSE 0.85847 (epoch 3) | 30 s | best so far 0.85847
[3/12] factor_regularization=4 learning_rate=0.002 bias_regularization=1 ...
    validation MSE 0.86079 (epoch 2) | 27 s | best so far 0.85847
[4/12] factor_regularization=4 learning_rate=0.002 bias_regularization=3 ...
    validation MSE 0.86068 (epoch 2) | 27 s | best so far 0.85847
[5/12] factor_regularization=8 learning_rate=0.001 bias_regularization=1 ...

## 6) SVD (k = 500, batches of 512)

In [10]:
svd_sweep = overnight_sweep("svd", {
    "user_bias_regularization": [1.0, 3.0],
    "item_bias_regularization": [3.0, 5.0],
    "factor_regularization": [4.0, 8.0, 16.0],
}, rerun_finalists_on_other_splits=True, n_finalists=2)

task3 / svd: 12 combinations + 4 finalist runs on split seeds [1, 2] | swept: {'user_bias_regularization': [1.0, 3.0], 'item_bias_regularization': [3.0, 5.0], 'factor_regularization': [4.0, 8.0, 16.0]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\svd_2026-11-08_10-26-46.json
[1/12] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=4 ...
    validation MSE 0.89203 (epoch 4) | 105 s | best so far 0.89203
[2/12] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=8 ...
    validation MSE 0.89144 (epoch 4) | 105 s | best so far 0.89144
[3/12] user_bias_regularization=1 item_bias_regularization=3 factor_regularization=16 ...
    validation MSE 0.89530 (epoch 6) | 126 s | best so far 0.89144
[4/12] user_bias_regularization=1 item_bias_regularization=5 factor_regularization=4 ...
    validation MSE 0.89164 (epoch 4) | 105 s | best so far 0.89144
[5/12] user_bias_regulari

## 7) Refinements around the winners: number of factors

In [11]:
svdpp_factors_sweep = overnight_sweep("svdpp", {
    **best_values(svdpp_sweep),
    "n_factors": [250, 500, 750, 1000],
    "ratings_per_batch": [8192, 16384],
})

task3 / svdpp: 8 combinations | swept: {'n_factors': [250, 500, 750, 1000], 'ratings_per_batch': [8192, 16384]} | constant: {'user_bias_regularization': 3.0, 'item_bias_regularization': 5.0, 'factor_regularization': 4.0, 'learning_rate': 0.001}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\svdpp_2026-11-08_10-56-15.json
[1/8] n_factors=250 ratings_per_batch=8192 ...
    validation MSE 0.88429 (epoch 3) | 32 s | best so far 0.88429
[2/8] n_factors=250 ratings_per_batch=16384 ...
    validation MSE 0.88628 (epoch 3) | 27 s | best so far 0.88429
[3/8] n_factors=500 ratings_per_batch=8192 ...
    validation MSE 0.88417 (epoch 2) | 31 s | best so far 0.88417
[4/8] n_factors=500 ratings_per_batch=16384 ...
    validation MSE 0.88415 (epoch 3) | 28 s | best so far 0.88415
[5/8] n_factors=750 ratings_per_batch=8192 ...
    validation MSE 0.88340 (epoch 2) | 37 s | best so far 0.88340
[6/8] n_factors=750 ratings_per_batch=16384 ...
    

In [12]:
svd_factors_sweep = overnight_sweep("svd", {
    **best_values(svd_sweep),
    "n_factors": [250, 500, 1000],
    "learning_rate": [2.5e-4, 5e-4],
})

task3 / svd: 6 combinations | swept: {'n_factors': [250, 500, 1000], 'learning_rate': [0.00025, 0.0005]} | constant: {'user_bias_regularization': 3.0, 'item_bias_regularization': 5.0, 'factor_regularization': 8.0}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\svd_2026-11-08_11-00-38.json
[1/6] n_factors=250 learning_rate=0.00025 ...
    validation MSE 0.89038 (epoch 5) | 69 s | best so far 0.89038
[2/6] n_factors=250 learning_rate=0.0005 ...
    validation MSE 0.89051 (epoch 3) | 56 s | best so far 0.89038
[3/6] n_factors=500 learning_rate=0.00025 ...
    validation MSE 0.88964 (epoch 4) | 105 s | best so far 0.88964
[4/6] n_factors=500 learning_rate=0.0005 ...
    validation MSE 0.89134 (epoch 3) | 95 s | best so far 0.88964
[5/6] n_factors=1000 learning_rate=0.00025 ...
    validation MSE 0.88996 (epoch 4) | 185 s | best so far 0.88964
[6/6] n_factors=1000 learning_rate=0.0005 ...
    validation MSE 0.89177 (epoch 2) | 148 s 

## 8) Until the deadline: random search for timeSVD++

In [13]:
# 360 combinations in random order -> stops at the deadline (a random sample of the grid)
time_svdpp_search = overnight_sweep("time_svdpp", {
    "factor_regularization": [4.0, 6.0, 8.0, 12.0, 16.0],
    "learning_rate": [5e-4, 1e-3, 2e-3, 4e-3],
    "n_factors": [250, 500, 750],
    "ratings_per_batch": [4096, 16384],
    "bias_regularization": [1.0, 3.0, 5.0],
}, random_order_seed=0)

task3 / time_svdpp: 360 combinations | swept: {'factor_regularization': [4.0, 6.0, 8.0, 12.0, 16.0], 'learning_rate': [0.0005, 0.001, 0.002, 0.004], 'n_factors': [250, 500, 750], 'ratings_per_batch': [4096, 16384], 'bias_regularization': [1.0, 3.0, 5.0]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task3\time_svdpp_2026-11-08_11-11-35.json
[1/360] factor_regularization=16 learning_rate=0.001 n_factors=500 ratings_per_batch=4096 bias_regularization=3 ...
    validation MSE 0.86170 (epoch 3) | 53 s | best so far 0.86170
[2/360] factor_regularization=12 learning_rate=0.002 n_factors=750 ratings_per_batch=4096 bias_regularization=3 ...
    validation MSE 0.86961 (epoch 1) | 48 s | best so far 0.86170
[3/360] factor_regularization=8 learning_rate=0.001 n_factors=250 ratings_per_batch=16384 bias_regularization=3 ...
    validation MSE 0.85942 (epoch 4) | 32 s | best so far 0.85942
[4/360] factor_regularization=4 learning_rat

## Overview

In [14]:
grid_search.keep_computer_awake(False)
print("finished", f"{datetime.now():%Y-%m-%d %H:%M}")
print_sweep_overview("task3")

finished 2026-11-08 13:59
bias_model_2026-11-08_09-58-29.json                finished         31 / 31 | best 0.91800 with user_bias_regularization=2 item_bias_regularization=5
item_knn_2026-11-08_10-00-13.json                  finished         33 / 33 | best 0.88855 with n_neighbors=200 prediction_shrinkage=0.3 similarity_shrinkage=50
svd_2026-11-08_10-26-46.json                       finished         16 / 16 | best 0.89422 with user_bias_regularization=3 item_bias_regularization=5 factor_regularization=8
svd_2026-11-08_11-00-38.json                       finished           6 / 6 | best 0.88964 with n_factors=500 learning_rate=0.00025
svdpp_2026-11-08_10-05-52.json                     finished         28 / 28 | best 0.88962 with user_bias_regularization=3 item_bias_regularization=5 factor_regularization=4 learning_rate=0.001
svdpp_2026-11-08_10-56-15.json                     finished           8 / 8 | best 0.88340 with n_factors=750 ratings_per_batch=8192
time_bias_model_2026-11-08_09-